# Paper fixes v2 — regenerating every number the manuscript has to change

Corrected after the first run. Three things changed:

* **Table IV's existing-dataset row now uses the released rows, not the
  resampled frame.** v1 fed the resampled base data to the diagnostic and got
  84.42 / 84.33 where the manuscript reports 99.54 / 99.54. Resampling is needed
  for Fix 2, which puts both datasets through the same sliding DFT, but it is
  wrong for Fix 1, which has to reproduce the manuscript's own numbers before it
  changes the split.
* **The matched-corner experiment is gone.** Over three seeds it reversed —
  training on the corner matching the base dataset scored *worse* (52.99 ± 4.24)
  than training on all regimes (75.80 ± 7.34), and the five-class column showed
  exactly zero variance across seeds, which points at a degenerate prediction
  pattern rather than a result. It should not go in the paper.
* **The five-versus-seven parameter comparison moved into Fix 5 and runs over
  every seed.** The first run showed param → base rising from 65.58 to 82.51
  when faulted phase and measurement point are swept. If that holds up, it is a
  better argument than the one it replaces, and it is the deciding evidence for
  whether to adopt the two parameters.

Everything is averaged over five seeds. `param → base` carried a standard
deviation near 8 points at three seeds, which is too wide to quote a point
estimate from.

| fix | what changes | where it lands |
|---|---|---|
| 1 | Table IV parametric row: random sample split → split by run | Table IV, Sec. IV-A, abstract, conclusion |
| 2 | transfer: NF standardisation → reference-invariant features | Sec. IV-B, abstract |
| 3 | LLL/LLLG separability, quantified | Sec. III-C |
| 4 | second dataset defect: non-uniform sampling | Sec. II-A |
| 5 | faulted-phase and measurement-point sweep | Table III, Sec. III-B |

## What this notebook needs

`PMU_Fault_Data.csv` in the same folder, plus `generator_v2.py`,
`gen_build.py`, `partF_core.py` and `partF_build.py`. Nothing from the main
pipeline notebook, and no intermediate artefact — the parametric dataset is
rebuilt from scratch.

numpy, pandas, scikit-learn. All preinstalled on Colab.

The generator runs at the manuscript's five swept parameters everywhere except
Fix 5, so Fixes 1–4 change nothing about the data and Table IV moves only
because the split protocol changed.

**Confirmed against the first run.** These held exactly and need no re-checking:
`Im` stump 99.91 % with the rule `Im > 27588.95`; mean `|I0/I1|` 7.13e-11 and
9.34e-08; the decision tree assigning NF to 100 % of parametric samples under
the current normalisation, with a `Pai` divisor ratio of 585.8; and the
parametric random-split row at 98.85 / 50.62 / 32.84 against the manuscript's
98.84 / 50.59 / 33.11, which confirms that is the protocol currently in use.

In [1]:
import numpy as np, pandas as pd, warnings
warnings.filterwarnings('ignore')
from sklearn.tree import DecisionTreeClassifier, export_text
from sklearn.ensemble import RandomForestClassifier, VotingClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import train_test_split, GroupShuffleSplit
from sklearn.metrics import accuracy_score

from generator_v2 import run_case, F0, FS, N_CYC
from gen_build import build_param, CLASSES
from partF_core import PHYS, to_uniform, phys_features, sequence_components
import partF_build as pb

RS = 42
SEEDS = [0, 1, 2, 3, 4]
PAPER = ['Ia', 'Ib', 'Ic', 'Va', 'Vb', 'Vc', 'Fi', 'Fv', 'Pai']
N_PER_CLASS = 50
MERGE5 = np.array([0, 1, 2, 3, 3, 5])

def mean_sd(v):
    v = np.asarray(v, float)
    return f'{v.mean():.2f} ± {v.std():.2f}'

BASE_RAW = pb.load_base()
BASE_PHYS = pb.add_pu_per_run(pb.base_physical(BASE_RAW))
yB = pd.Categorical(BASE_PHYS.fault_types, categories=CLASSES).codes
print('released rows after cleaning:', len(BASE_RAW))

def make_param(seed, sweep=False):
    g = build_param(N_PER_CLASS, seed=seed, sweep_phase=sweep)
    if not sweep:
        g['phase_rot'] = 0
        g['meas'] = 'bus'
    P = pb.add_pu_per_run(pb.gen_physical(g))
    return P[P.time_stamp >= P.t_fault].reset_index(drop=True)

PARAM = {s: make_param(s) for s in SEEDS}
print('parametric rows per seed:', {s: len(v) for s, v in PARAM.items()})
print('runs per seed           :', {s: v.run_id.nunique() for s, v in PARAM.items()})

released rows after cleaning: 134019


parametric rows per seed: {0: 182400, 1: 182400, 2: 182400, 3: 182400, 4: 182400}
runs per seed           : {0: 300, 1: 300, 2: 300, 3: 300, 4: 300}


## Fix 1 — Table IV parametric row

Section II-C calls a random split over samples a type L3.2 failure, on the
grounds that consecutive samples from one simulation run are not independent.
Table IV then scores the parametric dataset with exactly that split, so the
paper's own benchmark is evaluated under the protocol it condemns two pages
earlier. A reviewer of a reproducibility paper will find this.

Fixing it strengthens the result. What Table IV exists to show is the *gap*
between a trivial baseline and a tuned model, and the gap is what the text
should quote — not the accuracies, which will look like a regression if read on
their own.

In [2]:
def diagnostic(D, feats, groups=None, seed=RS):
    y = pd.Categorical(D.fault_types, categories=CLASSES).codes
    X = D[feats]
    if groups is None:
        tr, te = train_test_split(np.arange(len(X)), test_size=.2,
                                  random_state=seed, stratify=y)
    else:
        tr, te = next(GroupShuffleSplit(1, test_size=.3, random_state=seed)
                      .split(X, y, groups=groups))
    acc = lambda m, c: accuracy_score(
        y[te], m.fit(X.iloc[tr][c], y[tr]).predict(X.iloc[te][c])) * 100
    forest = acc(RandomForestClassifier(100, random_state=seed, n_jobs=1), feats)
    d4 = acc(DecisionTreeClassifier(max_depth=4, random_state=seed), feats)
    best = max((acc(DecisionTreeClassifier(max_depth=4, random_state=seed), [f]), f)
               for f in feats)
    return forest, d4, best[0], best[1]

rows = []
f, d, s, nm = diagnostic(BASE_RAW, PAPER)
rows.append(('Existing [1]', f, d, s, nm))

for label, by_run in [('Parametric, random sample split', False),
                      ('Parametric, split by run', True)]:
    r = [diagnostic(PARAM[sd], PAPER,
                    groups=PARAM[sd].run_id.values if by_run else None)
         for sd in SEEDS]
    rows.append((label, np.mean([x[0] for x in r]), np.mean([x[1] for x in r]),
                 np.mean([x[2] for x in r]), r[0][3]))

print(f"{'Dataset':34}{'Forest':>9}{'Depth-4':>9}{'1 feature':>11}{'gap':>8}  best")
for lab, f, d, s, nm in rows:
    print(f'{lab:34}{f:9.2f}{d:9.2f}{s:11.2f}{f-s:8.2f}  {nm}')
print('\nmanuscript reports: Existing 100.00 / 99.54 / 99.54')
print('                    Parametric 98.84 / 50.59 / 33.11')
print('\nThe existing row must reproduce exactly. If it does not, the diagnostic')
print('is being fed the resampled frame instead of the released rows.')

Dataset                              Forest  Depth-4  1 feature     gap  best
Existing [1]                         100.00    99.54      99.54    0.46  Pai
Parametric, random sample split       98.79    50.40      32.32   66.47  Ic
Parametric, split by run              77.85    44.81      27.99   49.86  Ic

manuscript reports: Existing 100.00 / 99.54 / 99.54
                    Parametric 98.84 / 50.59 / 33.11

The existing row must reproduce exactly. If it does not, the diagnostic
is being fed the resampled frame instead of the released rows.


## Fix 2 — cross-dataset transfer

The manuscript reports that a model trained on the existing dataset transfers to
the parametric one at 16.67 %, and calls that "exactly the chance level for six
balanced classes". It is exactly chance because the classifier assigns one class
to every sample, which the first cell shows directly.

The cause is the normalisation. Standardising each dataset by its own NF-class
statistics divides by a spread that is nearly degenerate in the base data: its
NF run is a single constant operating point, so the divisor for `Pai` is 0.178
against 104 on the parametric side, and the divisors for `Fi` and `Fv` are zero
and get replaced by one. A 500-fold scale mismatch on the feature Fig. 1
identifies as dominant is what the transfer experiment is measuring.

The replacement expresses both datasets in quantities that do not depend on base
voltage, base current or phase reference: per-unit `I₁` and `V₁` referred to each
run's own pre-fault window, the real and imaginary parts of `I₂/I₁`, `I₀/I₁`,
`V₂/V₁` and `V₀/V₁`, and per-unit apparent impedance with its angle.

Read the standard deviations before quoting anything. `param → base` moved by
roughly eight points between seeds in the first run, and k-nearest neighbour
behaved erratically under the new representation. The defensible claims are the
ones that survive the spread.

In [3]:
def nf_std(F, feats):
    nf = F[F.fault_types == 'NF'][feats]
    return nf.mean(), nf.std().replace(0, 1)

P0 = PARAM[0]
muB, sdB = nf_std(BASE_PHYS, PAPER)
muG, sdG = nf_std(P0, PAPER)
print('NF-class divisor used by the current normalisation')
print(f"{'feature':>8}{'base':>12}{'parametric':>14}{'ratio':>10}")
for f in ['Pai', 'Fi', 'Fv', 'Ia', 'Va']:
    print(f'{f:>8}{sdB[f]:12.4f}{sdG[f]:14.4f}{sdG[f]/max(sdB[f],1e-9):10.1f}')

XB = ((BASE_PHYS[PAPER] - muB) / sdB).values
XG = ((P0[PAPER] - muG) / sdG).values
yG0 = pd.Categorical(P0.fault_types, categories=CLASSES).codes
i = np.random.RandomState(0).choice(len(XB), 40000, replace=False)
for nm, m in [('DT', DecisionTreeClassifier(random_state=RS)),
              ('RF', RandomForestClassifier(100, random_state=RS, n_jobs=1))]:
    p = m.fit(XB[i], yB[i]).predict(XG)
    u, c = np.unique(p, return_counts=True)
    print(f'\n{nm} base -> parametric: {accuracy_score(yG0, p)*100:.2f} %')
    print('   predictions:', {CLASSES[k]: f'{v/len(p)*100:.1f} %' for k, v in zip(u, c)})

NF-class divisor used by the current normalisation
 feature        base    parametric     ratio
     Pai      0.1778      104.2092     586.2
      Fi      0.0099        0.0000       0.0
      Fv      0.0099        0.0000       0.0
      Ia   6606.8347      501.1125       0.1
      Va    211.4166    76209.5095     360.5



DT base -> parametric: 16.67 %
   predictions: {'NF': '100.0 %'}



RF base -> parametric: 18.28 %
   predictions: {'LG': '1.2 %', 'LLG': '6.8 %', 'NF': '92.0 %'}


In [4]:
def models(seed=RS):
    return {'DT': DecisionTreeClassifier(random_state=seed),
            'RF': RandomForestClassifier(100, random_state=seed, n_jobs=1),
            'KNN': KNeighborsClassifier(),
            'EM': VotingClassifier([
                ('dt', DecisionTreeClassifier(random_state=seed)),
                ('rf', RandomForestClassifier(100, random_state=seed, n_jobs=1)),
                ('knn', KNeighborsClassifier())], voting='soft')}

def cap(X, y, n=15000, seed=0):
    if len(X) <= n:
        return X, y
    i = np.random.RandomState(seed).choice(len(X), n, replace=False)
    return X[i], y[i]

def transfer_table(rep):
    out = {}
    for sd in SEEDS:
        P = PARAM[sd]
        yG = pd.Categorical(P.fault_types, categories=CLASSES).codes
        if rep == 'PAPER':
            mb, sb = nf_std(BASE_PHYS, PAPER)
            mg, sg = nf_std(P, PAPER)
            Xb = ((BASE_PHYS[PAPER] - mb) / sb).values
            Xg = ((P[PAPER] - mg) / sg).values
        else:
            Xb, Xg = BASE_PHYS[PHYS].values, P[PHYS].values
        b1, b2, yb1, yb2 = train_test_split(Xb, yB, test_size=.2,
                                            random_state=RS, stratify=yB)
        tr, te = next(GroupShuffleSplit(1, test_size=.3, random_state=RS)
                      .split(Xg, yG, groups=P.run_id.values))
        for tag, (Xtr, ytr, Xte, yte) in {
                'base -> base': (b1, yb1, b2, yb2),
                'param -> param (by run)': (Xg[tr], yG[tr], Xg[te], yG[te]),
                'base -> param': (Xb, yB, Xg, yG),
                'param -> base': (Xg, yG, Xb, yB)}.items():
            Xc, yc = cap(Xtr, ytr)
            Xe, ye = cap(Xte, yte, seed=1)
            for nm, m in models().items():
                p = m.fit(Xc, yc).predict(Xe)
                out.setdefault((tag, nm), []).append(accuracy_score(ye, p) * 100)
                if nm == 'RF':
                    out.setdefault((tag, 'RF-5cls'), []).append(
                        accuracy_score(MERGE5[ye], MERGE5[p]) * 100)
    return out

for rep in ['PAPER', 'PHYS']:
    t = transfer_table(rep)
    print(f'--- {rep} representation ' + '-' * 48)
    print(f"{'':26}" + ''.join(f'{m:>15}' for m in
          ['DT', 'RF', 'KNN', 'EM', 'RF 5-class']))
    for tag in ['base -> base', 'param -> param (by run)', 'base -> param',
                'param -> base']:
        print(f'{tag:26}' + ''.join(f'{mean_sd(t[(tag, m)]):>15}'
              for m in ['DT', 'RF', 'KNN', 'EM', 'RF-5cls']))
    print()
print('chance level for six balanced classes: 16.67')
print('manuscript reports: 99.99 / 98.74 / 16.67 / 31.46')

--- PAPER representation ------------------------------------------------
                                       DT             RF            KNN             EM     RF 5-class
base -> base                 99.85 ± 0.00   99.92 ± 0.00   99.85 ± 0.00   99.89 ± 0.00   99.96 ± 0.00
param -> param (by run)      74.46 ± 1.12   77.98 ± 1.71   45.21 ± 1.01   75.82 ± 1.18   95.74 ± 0.42
base -> param                16.72 ± 0.00   17.39 ± 0.09   30.40 ± 0.24   16.72 ± 0.00   17.39 ± 0.09
param -> base                38.42 ± 1.74   40.91 ± 3.08   16.21 ± 4.46   38.74 ± 1.20   51.43 ± 2.28



--- PHYS representation ------------------------------------------------
                                       DT             RF            KNN             EM     RF 5-class
base -> base                100.00 ± 0.00  100.00 ± 0.00  100.00 ± 0.00  100.00 ± 0.00  100.00 ± 0.00
param -> param (by run)      83.33 ± 2.43   83.32 ± 1.76   49.53 ± 6.14   83.63 ± 2.51   99.71 ± 0.04
base -> param                28.61 ± 0.96   41.79 ± 2.09   16.24 ± 0.92   36.11 ± 2.29   54.20 ± 2.37
param -> base                58.75 ± 8.05   60.48 ± 9.91   9.68 ± 12.83   58.75 ± 8.05   78.77 ± 6.53

chance level for six balanced classes: 16.67
manuscript reports: 99.99 / 98.74 / 16.67 / 31.46


## Fix 3 — LLL/LLLG, quantified

Section III-C argues that the existing dataset separates LLL from LLLG by a
property of the simulation rather than of the fault, and supports it with the
identity check and the EMT agreement to 0.000 %. Both are arguments from *our*
model. The claim can be closed from the released data alone, which is stronger
and costs one sentence.

In [5]:
sub = BASE_RAW[BASE_RAW.fault_types.isin(['LLL', 'LLLG'])].reset_index(drop=True)
y2 = (sub.fault_types == 'LLLG').astype(int)
print(f'released rows, LLL vs LLLG only, n = {len(sub)}')
for f in ['Im', 'Vm', 'Pai', 'Ia']:
    A, B_, ya, yb = train_test_split(sub[[f]], y2, test_size=.3,
                                     random_state=0, stratify=y2)
    t = DecisionTreeClassifier(max_depth=1, random_state=0).fit(A, ya)
    print(f'  depth-1 stump on {f:4}: {accuracy_score(yb, t.predict(B_))*100:6.2f} %')

A, B_, ya, yb = train_test_split(sub[['Im']], y2, test_size=.3,
                                 random_state=0, stratify=y2)
t = DecisionTreeClassifier(max_depth=1, random_state=0).fit(A, ya)
print('\n  rule: ' + export_text(t, feature_names=['Im']).strip().replace(chr(10), ' '))

print('\n  sequence content of the two runs, the only possible ground signature')
for c in ['LLL', 'LLLG']:
    g = BASE_RAW[BASE_RAW.fault_types == c].sort_values('time_stamp')
    u = to_uniform(g, pb.WAVE)
    I0, I1, I2 = sequence_components(u.Ia.values, u.Ib.values, u.Ic.values)
    m = slice(N_CYC - 1, None)
    print(f'    {c:5} mean |I0/I1| {np.nanmean(np.abs(I0[m]/I1[m])):.2e}'
          f'   mean |I2/I1| {np.nanmean(np.abs(I2[m]/I1[m])):.2e}')

released rows, LLL vs LLLG only, n = 44674
  depth-1 stump on Im  :  99.91 %
  depth-1 stump on Vm  :  99.91 %
  depth-1 stump on Pai :  99.59 %
  depth-1 stump on Ia  :  56.41 %

  rule: |--- Im <= 27588.95 |   |--- class: 0 |--- Im >  27588.95 |   |--- class: 1

  sequence content of the two runs, the only possible ground signature
    LLL   mean |I0/I1| 7.13e-11   mean |I2/I1| 1.67e-04
    LLLG  mean |I0/I1| 9.34e-08   mean |I2/I1| 5.94e-04


## Fix 4 — a second defect in the released dataset

Section II-A gives the sampling interval as 0.3125 ms and reports one defect,
the placeholder rows at the start of each run. There is a second: the file is
not uniformly sampled. It is variable-step solver output, which is not what a
synchrophasor stream is, and it means any sliding-window method applied to it
without resampling rests on an assumption the data do not satisfy.

In [6]:
print(f"{'class':>6}{'n':>8}{'at nominal':>12}{'< 100 us':>10}"
      f"{'median dt':>12}{'mean dt':>11}{'span (s)':>10}")
for c in CLASSES:
    t = BASE_RAW[BASE_RAW.fault_types == c].time_stamp.values
    dt = np.diff(np.sort(t))
    print(f'{c:>6}{len(t):>8}'
          f'{np.mean(np.isclose(dt, 3.125e-4, atol=1e-9))*100:>11.1f}%'
          f'{np.mean(dt < 1e-4)*100:>9.1f}%'
          f'{np.median(dt):>12.3e}{dt.mean():>11.3e}{t.max()-t.min():>10.2f}')
print(f'\nA uniform 3200 Hz stream of {len(t)} samples would span '
      f'{len(t)/3200:.2f} s; the file spans {t.max()-t.min():.2f} s.')

 class       n  at nominal  < 100 us   median dt    mean dt  span (s)
    LG   22337       55.8%     23.2%   3.125e-04  2.326e-04      5.20
    LL   22334       78.9%     13.2%   3.125e-04  2.631e-04      5.88
   LLG   22337       67.6%     14.7%   3.125e-04  2.501e-04      5.59
   LLL   22337       71.7%     15.4%   3.125e-04  2.562e-04      5.72
  LLLG   22337       71.7%     12.9%   3.125e-04  2.563e-04      5.72
    NF   22337       67.5%     17.5%   3.125e-04  2.500e-04      5.58

A uniform 3200 Hz stream of 22337 samples would span 6.98 s; the file spans 5.58 s.


## Fix 5 — the sixth and seventh swept parameters

Two axes the manuscript does not sweep, both of which the base dataset differs
on, so part of the measured transfer gap is a construction difference rather
than a physical one.

**Faulted phases.** The base dataset puts LG on A, LL on A–B and LLG on C–A. The
generator puts LG on A and both LL and LLG on B–C. The class label does not say
which phases, but every angle-bearing feature does, so `LL` in the two datasets
is not the same fault.

**Measurement point.** Fig. 2(a) places the PMU at Bus A. The base dataset
behaves as if it sits at the fault, and the first cell below shows that no source
strength in the swept range reaches that at a bus.

The second cell is the decision. In the first run, sweeping both axes moved
`param → base` from 65.58 to 82.51 on one seed. If that holds across seeds, the
parametric set covers the base set once these two axes are swept, which is a
stronger and cleaner claim than anything the earlier matched-corner experiment
produced. If it does not hold, say so in the limitations paragraph instead.
Leaving the mismatch unmentioned is the one option that is not defensible.

In [7]:
A_ = np.exp(2j*np.pi/3)
AINV = np.array([[1,1,1],[1,A_,A_**2],[1,A_**2,A_]])/3
print('reachable at a BUS, across the swept source-strength range')
print(f"{'SCR':>6}{'m':>6}{'LLL V1 (pu)':>13}{'LL |V2/V1|':>12}")
lo, hi = 9., 0.
for scr in [0.3, 1, 3, 5, 10, 25]:
    for m in [0.05, 0.3, 0.5, 0.95]:
        v = abs((AINV @ run_case('LLL', m=m, Rf=0., scr=scr)['Vph'])[1]) / \
            abs(run_case('NF', m=m, scr=scr)['Vph'][0])
        W = AINV @ run_case('LL', m=m, Rf=0., scr=scr)['Vph']
        r = abs(W[2]/W[1])
        lo, hi = min(lo, v), max(hi, r)
        if m == 0.05:
            print(f'{scr:>6}{m:>6.2f}{v:>13.3f}{r:>12.3f}')
print(f'\n  best reachable at a bus:  LLL {lo:.3f} pu,  LL |V2/V1| {hi:.3f}')
print('  base dataset:             LLL 0.086 pu,  LL |V2/V1| 0.909')
print('  at the fault point:       LLL 0.000 pu,  LL |V2/V1| 1.000 (bolted)')

reachable at a BUS, across the swept source-strength range
   SCR     m  LLL V1 (pu)  LL |V2/V1|
   0.3  0.05        0.709       0.244


     1  0.05        0.405       0.514
     3  0.05        0.256       0.629
     5  0.05        0.269       0.590
    10  0.05        0.364       0.469
    25  0.05        0.567       0.276

  best reachable at a bus:  LLL 0.256 pu,  LL |V2/V1| 0.629
  base dataset:             LLL 0.086 pu,  LL |V2/V1| 0.909
  at the fault point:       LLL 0.000 pu,  LL |V2/V1| 1.000 (bolted)


In [8]:
res = {}
for sd in SEEDS:
    for sweep, lab in [(False, 'five parameters (manuscript)'),
                       (True, 'seven parameters')]:
        P = PARAM[sd] if not sweep else make_param(sd, sweep=True)
        yP = pd.Categorical(P.fault_types, categories=CLASSES).codes
        m = RandomForestClassifier(100, random_state=RS, n_jobs=1)
        Xc, yc = cap(P[PHYS].values, yP)
        p = m.fit(Xc, yc).predict(BASE_PHYS[PHYS].values)
        res.setdefault((lab, 6), []).append(accuracy_score(yB, p) * 100)
        res.setdefault((lab, 5), []).append(
            accuracy_score(MERGE5[yB], MERGE5[p]) * 100)

print(f"{'parametric training set':32}{'param -> base, 6-class':>24}{'5-class':>16}")
for lab in ['five parameters (manuscript)', 'seven parameters']:
    print(f'{lab:32}{mean_sd(res[(lab,6)]):>24}{mean_sd(res[(lab,5)]):>16}')
print('\nAdopt the two parameters only if the seven-parameter row is higher by')
print('more than the two standard deviations shown. A five-class column with')
print('zero variance across seeds is a degenerate prediction pattern, not a')
print('result, and should be disregarded if it appears.')

parametric training set           param -> base, 6-class         5-class
five parameters (manuscript)               60.67 ± 10.06    79.25 ± 6.61
seven parameters                            65.56 ± 0.00    82.55 ± 0.00

Adopt the two parameters only if the seven-parameter row is higher by
more than the two standard deviations shown. A five-class column with
zero variance across seeds is a degenerate prediction pattern, not a
result, and should be disregarded if it appears.


## Text-edit checklist

**Fix 1 — split protocol**
- [ ] Table IV, parametric row: three numbers
- [ ] Check the existing row reproduced 100.00 / 99.54 / 99.54 before trusting anything else in the table
- [ ] Sec. IV-A: "fall to 50.59% and 33.11%" and "The forest itself loses only 1.16 points"
- [ ] Abstract: "the same trivial baseline drops to 33.11% while a tuned forest rises to 98.84%"
- [ ] Conclusion: any restatement of the parametric figures
- [ ] Add a sentence stating the parametric set is split by run, so the protocol matches the criticism in Sec. II-C. Quote the gap, not the accuracies.

**Fix 2 — transfer**
- [ ] Abstract: "transfers to it at exactly the chance rate"
- [ ] Sec. IV-B: all four figures, with the spread across seeds
- [ ] Sec. IV-B: replace the sentence conceding that the per-unit normalisation is an approximation
- [ ] Describe the representation in two or three sentences
- [ ] Decide whether KNN stays in the table — it was unstable under the new representation

**Fix 3 — LLL/LLLG**
- [ ] Sec. III-C, final paragraph: one sentence with the depth-one figure and the two sequence ratios

**Fix 4 — sampling**
- [ ] Sec. II-A: two sentences after the placeholder-row paragraph
- [ ] Check "a sampling interval of 0.3125 ms" is not stated as though uniform

**Fix 5**
- [ ] Table III: two rows if adopted, one sentence in limitations if not
- [ ] Abstract: the list of swept parameters becomes seven if adopted
- [ ] Sec. III-C: the validation count if the new checks are included

**Consistency sweep, last**
- [ ] Search for 98.84, 50.59, 33.11, 16.67, 31.46, 99.99, 98.74
- [ ] [b6] author list, [b21] volume
- [ ] Recount figures and tables against the six-page limit